# Initial checks

In [ ]:
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "src"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_path = DATA_DIR / "orders.csv"
missing_check_path = DATA_DIR / "oders.csv"
valid_output_path = OUTPUT_DIR / "processed_orders.csv"
rejected_output_path = OUTPUT_DIR / "rejected_orders.csv"

In [2]:
import pandas as pd

input_df = pd.read_csv(input_path)

print("Total source records excluding the header:", len(input_df))


duplicated_order_id = input_df[input_df['order_id'].duplicated()]
print("Number of duplicated order ids:", len(duplicated_order_id), "\nDuplicated Order ID:", duplicated_order_id['order_id'])

amount_converted_df = input_df.copy()
amount_converted_df['amount'] = pd.to_numeric(input_df['amount'], errors='coerce')
invalid_amount_df = amount_converted_df[(amount_converted_df['amount'] <= 0) | (amount_converted_df['amount'].isna())].copy()
print("Records with invalid amounts:", len(invalid_amount_df))

valid_status = ['completed', 'pending', 'cancelled']
invalid_status_df = input_df[~input_df['status'].str.lower().isin(valid_status)]
print("Records with invalid status:", len(invalid_status_df))

missing_customer_df = input_df[(input_df['customer'].isna()) | (input_df['customer'].str.strip() == "")].copy()
print("Records with missing customer name:", len(missing_customer_df))

Total source records excluding the header: 15
Number of duplicated order ids: 1 
Duplicated Order ID: 10    1002
Name: order_id, dtype: int64
Records with invalid amounts: 3
Records with invalid status: 1
Records with missing customer name: 1


# Pipeline build

In [ ]:
import pandas as pd
import logging 

logging.basicConfig(
    level = logging.INFO, 
    format = "%(asctime)s - %(levelname)s - %(message)s"
)

# Extract Function

def extract(input_path):
    try:
        logging.info("Extraction has started")
        df = pd.read_csv(input_path)
        logging.info("Extracted %d records from the %s file", len(df), input_path.name)
        return df
    except FileNotFoundError:
        logging.error("File Not found")
        return None

raw_df = extract(input_path)
if raw_df is not None:
    print(f"Input df with {raw_df.shape[0]} rows and {raw_df.shape[1]} columns:\n")
    display(raw_df)

# Check for missing or wrong file path/name
missing_df = extract(missing_check_path)
if missing_df is not None:
    print(f"Input df with {missing_df.shape[0]} rows and {missing_df.shape[1]} columns:\n")
    display(missing_df)

# Cleaning Function

def clean_data(df):
    if df is None:
        logging.warning("Cleaning skipped: Input Dataframe is None")
        return None

    logging.info("Data cleaning has been started")
    copy_df = df.copy()
    copy_df.columns = copy_df.columns.str.strip().str.lower()
    copy_df['customer'] = copy_df['customer'].str.strip().str.title()
    copy_df['status'] = copy_df['status'].str.strip().str.lower()
    copy_df['amount'] = pd.to_numeric(copy_df['amount'], errors='coerce')
    copy_df['order_date'] = pd.to_datetime(copy_df['order_date'], format="%m/%d/%Y", errors = 'coerce')
    logging.info("Data Cleaning has been completed with a total of %d records has been processed", len(copy_df))
    return copy_df 

cleaned_df = clean_data(raw_df)
if cleaned_df is not None:
    display(cleaned_df)
    print(cleaned_df.dtypes)

# Validation Function

def validate_data(df):
    if df is None:
        logging.warning("Validation skipped: Input Dataframe is None")
        return None, None

    logging.info("Data validation has been started")
    copy_df = df.copy()

    valid_order_id = (copy_df['order_id'].notna()) & (copy_df['order_id'].astype('string').str.strip() != "")
    duplicate_orders = ~copy_df['order_id'].duplicated(keep=False)
    valid_customer_name = (copy_df['customer'].notna()) & (copy_df['customer'].str.strip() != "")
    valid_amount = (copy_df['amount'] > 0) & (copy_df['amount'].notna())
    valid_status = (copy_df['status'].isin(['completed', 'pending', 'cancelled']))
    valid_date = (copy_df['order_date'].notna())
    
    valid_mask = valid_order_id & duplicate_orders & valid_customer_name & valid_amount & valid_status & valid_date
    valid_df = copy_df[valid_mask].reset_index(drop=True)
    rejected_df = copy_df[~valid_mask].reset_index(drop=True)

    logging.info("Total source records: %d", len(copy_df))
    logging.info("Total valid records: %d", len(valid_df))
    logging.info("Total rejected records: %d", len(rejected_df))

    logging.info("Data validation has been completed successfully")
    if len(valid_df) + len(rejected_df) == len(copy_df):
        logging.info("Valid and Rejected records sum matches the input records count")
        return valid_df, rejected_df
    logging.error("Valid and Rejected records sum doesn't match the input records count")
    return None, None

valid_df, rejected_df = validate_data(cleaned_df)

display(valid_df)
display(rejected_df)

print("Source:", len(cleaned_df))
print("Valid:", len(valid_df))
print("Rejected:", len(rejected_df))

# Transformation Function

def transform_data(df):
    if df is None:
        logging.warning("Transformation skipped: Input Dataframe is None")
        return None

    logging.info("Data transformation has been started")
    copy_df = df.copy()

    copy_df['tax'] = copy_df['amount']*0.05
    copy_df['total_amount'] = copy_df['amount'] + copy_df['tax']
    logging.info("Transformation is completed with a total of %d records transformed", len(copy_df))
    return copy_df

transformed_df = transform_data(valid_df)

display(transformed_df)
print("Transformed records:", len(transformed_df))

# Loading Function

def load_data(df, df_name, output_path):
    if df is None:
        logging.error("Loading skipped: Input Dataframe is None")
        return False
    try: 
        logging.info("Data loading has been started")
        df.to_csv(output_path, index=False)
        logging.info("The %s with %d records has been loaded to the path %s", df_name, len(df), output_path)
        return True
    except OSError: 
        logging.exception("Path not found")
        return False

valid_loaded = load_data(transformed_df, "Transformed Dataframe", valid_output_path)
rejected_loaded = load_data(rejected_df, "Rejected Dataframe", rejected_output_path)

print("Valid output saved:", valid_loaded)
print("Rejected output saved:", rejected_loaded)

2026-10-23 14:15:08,342 - INFO - Extraction has started
2026-10-23 14:15:08,348 - INFO - Extracted 15 records from the orders.csv file


Input df with 15 rows and 5 columns:



,order_id,customer,amount,status,order_date
0,1001,Naresh,2500,completed,10/1/2026
1,1002,Arun,1800,completed,10/2/2026
2,1003,Priya,3200,pending,10/3/2026
3,1004,Karthik,0,completed,10/4/2026
4,1005,Divya,1500,cancelled,10/5/2026
5,1006,Meena,invalid,completed,10/6/2026
6,1007,Rahul,4200,completed,10/7/2026
7,1008,NaN,2100,completed,10/8/2026
8,1009,Vijay,900,unknown,10/9/2026
9,1010,Anu,2800,completed,10/10/2026


2026-10-23 14:15:08,381 - INFO - Extraction has started
2026-10-23 14:15:08,383 - ERROR - File Not found
2026-10-23 14:15:08,385 - INFO - Data cleaning has been started
2026-10-23 14:15:08,398 - INFO - Data Cleaning has been completed with a total of 15 records has been processed


,order_id,customer,amount,status,order_date
0,1001,Naresh,2500.0,completed,2026-10-01
1,1002,Arun,1800.0,completed,2026-10-02
2,1003,Priya,3200.0,pending,2026-10-03
3,1004,Karthik,0.0,completed,2026-10-04
4,1005,Divya,1500.0,cancelled,2026-10-05
5,1006,Meena,NaN,completed,2026-10-06
6,1007,Rahul,4200.0,completed,2026-10-07
7,1008,NaN,2100.0,completed,2026-10-08
8,1009,Vijay,900.0,unknown,2026-10-09
9,1010,Anu,2800.0,completed,2026-10-10


...

2026-10-23 14:15:08,458 - INFO - Data transformation has been started
2026-10-23 14:15:08,468 - INFO - Transformation is completed with a total of 8 records transformed


Source: 15
Valid: 8
Rejected: 7


,order_id,customer,amount,status,order_date,tax,total_amount
0,1001,Naresh,2500.0,completed,2026-10-01,125.0,2625.0
1,1003,Priya,3200.0,pending,2026-10-03,160.0,3360.0
2,1005,Divya,1500.0,cancelled,2026-10-05,75.0,1575.0
3,1007,Rahul,4200.0,completed,2026-10-07,210.0,4410.0
4,1010,Anu,2800.0,completed,2026-10-10,140.0,2940.0
5,1011,Sanjay,3500.0,completed,2026-10-11,175.0,3675.0
6,1012,Latha,1200.0,pending,2026-10-12,60.0,1260.0
7,1014,Deepa,2600.0,completed,2026-10-14,130.0,2730.0


2026-10-23 14:15:08,487 - INFO - Data loading has been started
2026-10-23 14:15:08,503 - INFO - The Transformed Dataframe with 8 records has been loaded to the path c:\Naresh Kumar S\Learnings\Python\Practice code\python-ecommerce-etl\output\processed_orders.csv
2026-10-23 14:15:08,503 - INFO - Data loading has been started
2026-10-23 14:15:08,507 - INFO - The Rejected Dataframe with 7 records has been loaded to the path c:\Naresh Kumar S\Learnings\Python\Practice code\python-ecommerce-etl\output\rejected_orders.csv


Transformed records: 8
Valid output saved: True
Rejected output saved: True


In [4]:
# PostgreSQL Integration

import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "src"
    else Path.cwd()
)

env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path)

def get_db_engine():
    required_vars = [
        "DB_HOST",
        "DB_PORT",
        "DB_NAME",
        "DB_USER",
        "DB_PASSWORD",
    ]

    missing = [key for key in required_vars if not os.getenv(key)]
    if missing:
        raise ValueError(f"Missing environment variables: {missing}")

    db_url = URL.create(
        drivername="postgresql+psycopg2",
        username=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        host=os.environ["DB_HOST"],
        port=int(os.environ["DB_PORT"]),
        database=os.environ["DB_NAME"],
    )

    return create_engine(db_url, pool_pre_ping=True)

engine = get_db_engine()

# Loading Data to PostgreSQL as table

def load_to_postgres(df, table_name, engine):
    if df is None:
        logging.warning("Database load skipped: DataFrame is None")
        return False

    try:
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",
            index=False
        )
        logging.info(
            "Loaded %d records into PostgreSQL table %s",
            len(df), table_name
        )
        return True

    except Exception:
        logging.exception("Failed to load table %s", table_name)
        return False

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("PostgreSQL connection successful:", result.scalar())


PostgreSQL connection successful: 1


In [5]:
# Orchestration Function

def run_pipeline(input_path, valid_output_path, rejected_output_path):
    logging.info("Pipeline has been started")
    raw_df = extract(input_path)
    if raw_df is not None:
        cleaned_df = clean_data(raw_df)
        if cleaned_df is not None:
            valid_df, rejected_df = validate_data(cleaned_df)
            if valid_df is not None:
                transformed_df = transform_data(valid_df)
                if transformed_df is not None and rejected_df is not None:
                    valid_loaded = load_data(transformed_df, "Transformed Dataframe", valid_output_path)
                    reject_loaded = load_data(rejected_df, "Rejected Dataframe", rejected_output_path)
                    
                    if not valid_loaded:
                        logging.warning("Valid Orders is not loaded")
                    if not reject_loaded:
                        logging.warning("Reject Orders is not loaded")

                    if valid_loaded and reject_loaded:
                        valid_db_loaded = load_to_postgres(transformed_df, "processed_orders", engine)
                        reject_db_loaded = load_to_postgres(rejected_df, "rejected_orders", engine)
                        if valid_db_loaded and reject_db_loaded:
                            logging.info("Pipeline has been completed successfully")
                            return True

                        if not valid_db_loaded:
                            logging.warning("Valid Orders is not loaded to PostgreSQL")
                        if not reject_db_loaded:
                            logging.warning("Reject Orders is not loaded to PostgreSQL")

    logging.error("Pipeline has been failed")
    return False
orchestration = run_pipeline(
    input_path,
    valid_output_path,
    rejected_output_path
)
print("Pipeline Completed:", orchestration)

2026-10-23 14:15:17,695 - INFO - Pipeline has been started
2026-10-23 14:15:17,696 - INFO - Extraction has started
2026-10-23 14:15:17,700 - INFO - Extracted 15 records from the orders.csv file
2026-10-23 14:15:17,701 - INFO - Data cleaning has been started
2026-10-23 14:15:17,705 - INFO - Data Cleaning has been completed with a total of 15 records has been processed
2026-10-23 14:15:17,708 - INFO - Data validation has been started
2026-10-23 14:15:17,712 - INFO - Total source records: 15
2026-10-23 14:15:17,713 - INFO - Total valid records: 8
2026-10-23 14:15:17,714 - INFO - Total rejected records: 7
2026-10-23 14:15:17,714 - INFO - Data validation has been completed successfully
2026-10-23 14:15:17,715 - INFO - Valid and Rejected records sum matches the input records count
2026-10-23 14:15:17,715 - INFO - Data transformation has been started
2026-10-23 14:15:17,717 - INFO - Transformation is completed with a total of 8 records transformed
2026-10-23 14:15:17,717 - INFO - Data loading

Pipeline Completed: True


In [6]:
# Check tables created

engine = get_db_engine()

try:
    with engine.connect() as connection:
        processed_count = connection.execute(
            text("SELECT COUNT(*) FROM processed_orders")
        ).scalar_one()

        rejected_count = connection.execute(
            text("SELECT COUNT(*) FROM rejected_orders")
        ).scalar_one()

        print("Processed records:", processed_count)
        print("Rejected records:", rejected_count)

finally:
    engine.dispose()
    

Processed records: 8
Rejected records: 7
